In [1]:
import sys
!{sys.executable} -m pip install pybullet pillow numpy


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pybullet as p
import pybullet_data
import numpy as np
import json
import os
from PIL import Image
import time

# --- Setup ---
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

# Load objects
p.loadURDF("plane.urdf")
sphere = p.loadURDF("sphere2.urdf", basePosition=[0, 0, 2])


# --- Camera setup ---
cameras = [
    {"eye": [0, -2, 1.5], "target": [0, 0, 0.5], "up": [0, 0, 1]},
    {"eye": [2,  0, 1.5], "target": [0, 0, 0.5], "up": [0, 0, 1]},
    {"eye": [-2, 0, 1.5], "target": [0, 0, 0.5], "up": [0, 0, 1]},
]

# Precompute view matrices (don't recompute every step)
WIDTH, HEIGHT = 320, 240  # smaller resolution, faster saves
proj_matrix = p.computeProjectionMatrixFOV(
    fov=60, aspect=WIDTH/HEIGHT, nearVal=0.1, farVal=100
)
view_matrices = [
    p.computeViewMatrix(cam["eye"], cam["target"], cam["up"])
    for cam in cameras
]

for cam_idx in range(len(cameras)):
    os.makedirs(f"frames/cam{cam_idx}", exist_ok=True)

# --- Simulation loop ---
states = []
RENDER_EVERY = 25  # only render every Nth frame, reduces frame count by 2x

for i in range(500):
    p.stepSimulation()
        # Only save frames every Nth step, don't sleep
    time.sleep(1./240.)
   

    # Always log state
    pos, orn = p.getBasePositionAndOrientation(sphere)
    vel, ang_vel = p.getBaseVelocity(sphere)
    states.append({
        "step": i,
        "pos": list(pos),
        "orn": list(orn),
        "vel": list(vel),
        "ang_vel": list(ang_vel)
    })

    # Only render every Nth step
    if i % RENDER_EVERY != 0:
        continue

    for cam_idx, view_matrix in enumerate(view_matrices):
        _, _, rgb, _, _ = p.getCameraImage(
            width=WIDTH,
            height=HEIGHT,
            viewMatrix=view_matrix,
            projectionMatrix=proj_matrix,
            renderer=p.ER_TINY_RENDERER
        )
        rgb_array = np.array(rgb, dtype=np.uint8).reshape(HEIGHT, WIDTH, 4)
        img = Image.fromarray(rgb_array[:, :, :3])
        img.save(f"frames/cam{cam_idx}/frame_{i:04d}.png")

# --- Save ground truth ---
with open("trajectory.json", "w") as f:
    json.dump(states, f, indent=2)

p.disconnect()
print("Done.")

pybullet build time: May  3 2026 11:26:45


Version = 4.1 Metal - 83
Vendor = Apple
Renderer = Apple M1 Max
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started


TypeError: cannot unpack non-iterable NoneType object

: 